# 02 - Dataset2 Selected RBF SVM Training

Bu notebook, Dataset2 için seçilen patch/feature kombinasyonları üzerinde RBF SVM modellerini eğitir veya mevcut model dosyalarını yükler.

Amaç, daha pahalı model ailesini yalnızca seçilen kombinasyonlarda çalıştırmak ve sonraki test aşamalarında kullanılacak model registry tablolarını oluşturmaktır.


## 1. Kütüphaneler

Bu bölümde model eğitimi, GridSearchCV, metrik hesaplama, parquet okuma ve model kaydetme için kullanılan kütüphaneler yüklenir.


In [ ]:
from pathlib import Path
from datetime import datetime
import gc
import json
import time
import traceback
import warnings

import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)


## 2. Ayarlar

Dataset, algoritma, hiperparametre aralıkları, split bilgileri ve overwrite davranışı tanımlanır.


In [ ]:
# -----------------------------------------------------------------------------
# Runtime settings
# -----------------------------------------------------------------------------
RANDOM_STATE = 42
CV_FOLDS = 3
SCORING = "f1"
N_JOBS = -1
RF_ESTIMATOR_N_JOBS = 1

DEBUG_MODE = False
DEBUG_MAX_MODELS = 1
DEBUG_FILTER_ENABLED = False
DEBUG_FILTER_ROWS = [
    {"dataset_name": "dataset2_honeybee", "patch_set": "centered48_balanced", "feature_set": "hsv_lbp_only"},
]

OVERWRITE_MODELS = False
OVERWRITE_TABLES = False
OVERWRITE_FIGURES = False

ACTIVE_DATASETS = ["dataset2_honeybee"]

DATASET_CONFIGS = {
    "dataset1_varroa": {"dataset_short": "d1", "display_name": "Dataset1 Varroa"},
    "dataset2_honeybee": {"dataset_short": "d2", "display_name": "Dataset2 Honeybee"},
}

SPLIT_COLUMN = "split"
LABEL_COLUMN = "patch_label"
TRAIN_SPLIT = "train"
VALID_SPLIT = "valid"
TEST_SPLIT = "test"

EXPECTED_SELECTED_COMBINATIONS_TOTAL = 9
EXPECTED_SELECTED_COMBINATIONS_PER_DATASET = 9

NOTEBOOK_NAME = "02_dataset2_selected_rbf_svm_training"
ALGORITHM_NAME = "rbf_svm"
ALGORITHM_SHORT = "rbfsvm"
ALGORITHM_DISPLAY_NAME = "RBF SVM"
ALGORITHM_ORDER_PREFIX = "02"
HYPERPARAMETER_GRID_NAME = "rbf_svm_stage06_selected_v1_from_previous_svm_notebook"
PIPELINE_DESCRIPTION = "standard_scaler + svc_rbf"
SCALER_USED = True
SCORE_METHOD = "decision_function"
MODEL_FOLDER_NAME = "02_selected_rbf_svm"

# Adapted from the previous SVM notebook's RBF block.
PARAM_GRID = {
    "model__C": [0.01, 0.1, 1.0, 10.0],
    "model__gamma": ["scale", 0.01, 0.001],
}


## 3. Yardımcı Fonksiyonlar

Klasör oluşturma, dosya kaydetme, proje içi yol çözme ve tablo gösterme yardımcıları hazırlanır.


In [ ]:
def timestamp_now():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def ensure_dir(path):
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def log_output(message):
    print(message)


def log_section(title):
    print()
    print("=" * 80)
    print(title)
    print("=" * 80)


def log_dataframe(title, df, max_rows=20):
    log_section(title)
    print(f"Shape: {df.shape}")
    display(df.head(max_rows))


def save_dataframe_csv(df, path, overwrite=False, note=""):
    path = Path(path)
    ensure_dir(path.parent)

    if path.exists() and not overwrite:
        print(f"[SKIP] Existing CSV kept: {path}")
        try:
            return pd.read_csv(path)
        except pd.errors.EmptyDataError:
            print(f"[WARNING] Existing CSV is empty; current dataframe kept in memory: {path}")
            return df

    if path.exists() and overwrite:
        print(f"[OVERWRITE] Replacing CSV: {path}")

    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"[SAVE] CSV saved: {path}")
    return df


def save_model(payload, path, overwrite=False, note=""):
    path = Path(path)
    ensure_dir(path.parent)

    if path.exists() and not overwrite:
        print(f"[SKIP] Existing model kept: {path}")
        return "loaded_existing"

    if path.exists() and overwrite:
        print(f"[OVERWRITE] Replacing model: {path}")

    joblib.dump(payload, path)
    print(f"[SAVE] Model saved: {path}")
    return "saved"


def resolve_project_root(start_path=None):
    if start_path is None:
        start_path = Path.cwd()
    start_path = Path(start_path).resolve()

    candidates = [start_path] + list(start_path.parents)
    for candidate in candidates:
        if (candidate / "data").exists() and (candidate / "approaches").exists():
            return candidate

    raise FileNotFoundError(
        "Proje kökü bulunamadı. Notebook'u proje klasörü içinde çalıştırdığından emin ol."
    )


def normalize_path_from_csv(value, project_root):
    if value is None or pd.isna(value):
        return None

    raw = str(value).strip()
    path = Path(raw)
    if path.exists():
        return path

    project_root = Path(project_root)
    normalized = raw.replace(chr(92), "/")

    for marker in ["data/features/", "outputs/models/", "approaches/"]:
        if marker in normalized:
            suffix = normalized.split(marker, 1)[1]
            candidate = project_root / marker.rstrip("/") / suffix
            if candidate.exists():
                return candidate

    if raw.endswith(".parquet"):
        dataset_name = ACTIVE_DATASETS[0]
        candidate = project_root / "data" / "features" / dataset_name / Path(raw).name
        if candidate.exists():
            return candidate

    if raw.endswith(".joblib"):
        filename = Path(raw).name
        for dataset_name in ACTIVE_DATASETS:
            for folder_name in ["01_linear_svm", "02_selected_rbf_svm", "03_selected_random_forest"]:
                candidate = project_root / "outputs" / "models" / dataset_name / folder_name / "models" / filename
                if candidate.exists():
                    return candidate
                candidate = project_root / "outputs" / "models" / dataset_name / folder_name / filename
                if candidate.exists():
                    return candidate

    return path


def relative_path(path):
    path = Path(path)

    try:
        return path.resolve().relative_to(PROJECT_ROOT.resolve()).as_posix()
    except Exception:
        return path.as_posix()


def get_feature_columns(df):
    excluded_columns = {
        SPLIT_COLUMN,
        LABEL_COLUMN,
        "patch_id",
        "source_image_path",
        "source_label_path",
        "source_dataset",
        "dataset_name",
        "patch_set",
        "ratio_variant",
        "feature_set",
    }

    return [
        column
        for column in df.columns
        if column not in excluded_columns and pd.api.types.is_numeric_dtype(df[column])
    ]


def compute_binary_metrics(y_true, y_pred, score_values=None):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    return {
        "valid_accuracy": accuracy_score(y_true, y_pred),
        "valid_balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "valid_precision": precision_score(y_true, y_pred, zero_division=0),
        "valid_recall": recall_score(y_true, y_pred, zero_division=0),
        "valid_f1": f1_score(y_true, y_pred, zero_division=0),
        "valid_true_negative": tn,
        "valid_false_positive": fp,
        "valid_false_negative": fn,
        "valid_true_positive": tp,
    }


print("[OK] Yardımcı fonksiyonlar hazır.")


## 4. Dosya Yolları

Seçili aday tablosu, notebook çıktı klasörleri, ortak özet klasörü ve model klasörleri tanımlanır.


In [ ]:
PROJECT_ROOT = resolve_project_root()
APPROACH_DIR = PROJECT_ROOT / "approaches" / "traditional_ml"
STAGE_NAME = "06_selected_model_training"

SELECTED_CANDIDATES_PATH = (
    APPROACH_DIR
    / "inputs"
    / STAGE_NAME
    / "selected_linear_candidates_all.csv"
)

OUTPUT_DIR = APPROACH_DIR / "outputs" / STAGE_NAME / NOTEBOOK_NAME
TABLES_DIR = ensure_dir(OUTPUT_DIR / "tables")
FIGURES_DIR = ensure_dir(OUTPUT_DIR / "figures")
CV_RESULTS_DIR = ensure_dir(OUTPUT_DIR / "cv_results")

MODEL_OUTPUT_ROOTS = {
    dataset_name: PROJECT_ROOT
    / "outputs"
    / "models"
    / "selected_classification_model_pool"
    / dataset_name
    / ALGORITHM_NAME
    for dataset_name in ACTIVE_DATASETS
}

for model_root in MODEL_OUTPUT_ROOTS.values():
    ensure_dir(model_root)

if not SELECTED_CANDIDATES_PATH.exists():
    raise FileNotFoundError(f"Selected candidates file not found: {SELECTED_CANDIDATES_PATH}")

log_output(f"PROJECT_ROOT = {PROJECT_ROOT}")
log_output(f"SELECTED_CANDIDATES_PATH = {SELECTED_CANDIDATES_PATH}")
log_output(f"OUTPUT_DIR = {OUTPUT_DIR}")
for dataset_name, model_root in MODEL_OUTPUT_ROOTS.items():
    log_output(f"MODEL_OUTPUT_ROOT[{dataset_name}] = {model_root}")

## 5. Seçili Aday Planı

Önceki validasyon adımında seçilen patch/feature kombinasyonları okunur ve aktif dataset için filtrelenir.


In [ ]:
selected_candidates = pd.read_csv(SELECTED_CANDIDATES_PATH)

if "selected_for_stage06" in selected_candidates.columns:
    selected_candidates = selected_candidates[selected_candidates["selected_for_stage06"].astype(bool)].copy()

selected_candidates = selected_candidates[selected_candidates["dataset_name"].isin(ACTIVE_DATASETS)].copy()
selected_candidates = selected_candidates.reset_index(drop=True)

if len(selected_candidates) != EXPECTED_SELECTED_COMBINATIONS_TOTAL:
    raise ValueError(f"Expected {EXPECTED_SELECTED_COMBINATIONS_TOTAL} selected rows, found {len(selected_candidates)}.")

for dataset_name in ACTIVE_DATASETS:
    count = int((selected_candidates["dataset_name"] == dataset_name).sum())
    if count != EXPECTED_SELECTED_COMBINATIONS_PER_DATASET:
        raise ValueError(f"{dataset_name}: expected 9 rows, found {count}.")

selected_candidates["source_feature_file_path_resolved"] = selected_candidates["source_feature_file_path"].apply(
    lambda value: relative_path(normalize_path_from_csv(value, PROJECT_ROOT))
)

if "source_linear_svm_model_path" in selected_candidates.columns:
    selected_candidates["source_linear_svm_model_path_resolved"] = selected_candidates["source_linear_svm_model_path"].apply(
        lambda value: relative_path(normalize_path_from_csv(value, PROJECT_ROOT)) if pd.notna(value) and str(value).strip() else ""
    )
else:
    selected_candidates["source_linear_svm_model_path"] = ""
    selected_candidates["source_linear_svm_model_path_resolved"] = ""

log_dataframe("Selected Stage 06 Candidate Plan", selected_candidates, max_rows=30)
selected_candidates = save_dataframe_csv(selected_candidates, TABLES_DIR / "selected_stage06_candidates_used.csv", overwrite=OVERWRITE_TABLES)


## 6. Eğitim Konfigürasyonu

Algoritma ayarları ve hiperparametre grid özeti tablo olarak kaydedilir.


In [ ]:
config_df = pd.DataFrame([
    {"setting": "algorithm_name", "value": ALGORITHM_NAME},
    {"setting": "pipeline", "value": PIPELINE_DESCRIPTION},
    {"setting": "scaler_used", "value": SCALER_USED},
    {"setting": "score_method", "value": SCORE_METHOD},
    {"setting": "class_weight", "value": "None"},
    {"setting": "cv_folds", "value": CV_FOLDS},
    {"setting": "scoring", "value": SCORING},
    {"setting": "n_jobs", "value": N_JOBS},
    {"setting": "hyperparameter_grid_name", "value": HYPERPARAMETER_GRID_NAME},
])

log_dataframe(f"{ALGORITHM_DISPLAY_NAME} Configuration", config_df)

grid_rows = []
if ALGORITHM_NAME == "rbf_svm":
    for c_value in PARAM_GRID["model__C"]:
        for gamma_value in PARAM_GRID["model__gamma"]:
            grid_rows.append({"C": c_value, "gamma": gamma_value})
elif ALGORITHM_NAME == "random_forest":
    for n_estimators in PARAM_GRID["model__n_estimators"]:
        for max_depth in PARAM_GRID["model__max_depth"]:
            for min_samples_leaf in PARAM_GRID["model__min_samples_leaf"]:
                grid_rows.append({
                    "n_estimators": n_estimators,
                    "max_depth": max_depth,
                    "min_samples_leaf": min_samples_leaf,
                })

grid_df = pd.DataFrame(grid_rows)
log_dataframe(f"{ALGORITHM_DISPLAY_NAME} Hyperparameter Grid", grid_df, max_rows=50)

save_dataframe_csv(config_df, TABLES_DIR / "model_training_config.csv", overwrite=OVERWRITE_TABLES)
save_dataframe_csv(grid_df, TABLES_DIR / "hyperparameter_grid_summary.csv", overwrite=OVERWRITE_TABLES)


## 7. Eğitim Planı

Seçili adaylardan model kimlikleri, feature yolları ve model çıktı yolları oluşturulur.


In [ ]:
def build_training_plan(selected_df):
    rows = []

    for dataset_name in ACTIVE_DATASETS:
        dataset_df = selected_df[selected_df["dataset_name"] == dataset_name].copy().reset_index(drop=True)
        dataset_short = DATASET_CONFIGS[dataset_name]["dataset_short"]
        model_root = MODEL_OUTPUT_ROOTS[dataset_name]

        for idx, row in dataset_df.iterrows():
            model_index = idx + 1
            model_id = f"{dataset_short}_{ALGORITHM_SHORT}_{model_index:03d}"
            model_filename = f"{model_id}__{row['patch_set']}__{row['feature_set']}.joblib"

            rows.append({
                "dataset_name": dataset_name,
                "dataset_short": dataset_short,
                "model_index": model_index,
                "model_id": model_id,
                "algorithm_name": ALGORITHM_NAME,
                "algorithm_short": ALGORITHM_SHORT,
                "patch_set": row["patch_set"],
                "patch_size": row.get("patch_size", np.nan),
                "ratio_variant": row.get("ratio_variant", ""),
                "feature_set": row["feature_set"],
                "feature_family": row.get("feature_family", ""),
                "selection_group": row.get("selection_group", ""),
                "selection_reason": row.get("selection_reason", ""),
                "source_feature_file_path": row["source_feature_file_path"],
                "source_feature_file_path_resolved": row["source_feature_file_path_resolved"],
                "source_linear_svm_model_path": row.get("source_linear_svm_model_path", ""),
                "source_linear_svm_model_path_resolved": row.get("source_linear_svm_model_path_resolved", ""),
                "model_file_path": relative_path(model_root / model_filename),
            })

    plan_df = pd.DataFrame(rows)

    if DEBUG_MODE:
        if DEBUG_FILTER_ENABLED:
            parts = []
            for filter_row in DEBUG_FILTER_ROWS:
                mask = np.ones(len(plan_df), dtype=bool)
                for key, value in filter_row.items():
                    mask &= plan_df[key].eq(value)
                parts.append(plan_df[mask])
            plan_df = pd.concat(parts, ignore_index=True) if parts else plan_df.iloc[0:0].copy()
        else:
            plan_df = plan_df.head(DEBUG_MAX_MODELS).copy()

        log_output(f"DEBUG_MODE=True. Training plan reduced to {len(plan_df)} model(s).")

    return plan_df.reset_index(drop=True)


training_plan = build_training_plan(selected_candidates)
log_dataframe(f"Stage 06 {ALGORITHM_DISPLAY_NAME} Training Plan", training_plan, max_rows=40)
save_dataframe_csv(training_plan, TABLES_DIR / "training_plan.csv", overwrite=OVERWRITE_TABLES)


## 8. Model Eğitim Yardımcıları

Estimator, skor üretimi, metrik hesaplama ve tek model eğitimi/yükleme fonksiyonları tanımlanır.


In [ ]:
def make_estimator():
    if ALGORITHM_NAME == "rbf_svm":
        return Pipeline([
            ("scaler", StandardScaler()),
            ("model", SVC(kernel="rbf", probability=False, class_weight=None, random_state=RANDOM_STATE)),
        ])

    if ALGORITHM_NAME == "random_forest":
        return Pipeline([
            ("model", RandomForestClassifier(
                random_state=RANDOM_STATE,
                n_jobs=RF_ESTIMATOR_N_JOBS,
                class_weight=None,
            )),
        ])

    raise ValueError(f"Unsupported algorithm: {ALGORITHM_NAME}")


def extract_model_score(model, X):
    if SCORE_METHOD == "decision_function":
        return model.decision_function(X)
    if SCORE_METHOD == "predict_proba_positive_class":
        return model.predict_proba(X)[:, 1]
    raise ValueError(f"Unsupported score method: {SCORE_METHOD}")


def train_one_selected_model(plan_row):
    start_time = time.time()

    model_id = plan_row["model_id"]
    dataset_name = plan_row["dataset_name"]
    model_path = Path(plan_row["model_file_path"])
    feature_path = Path(plan_row["source_feature_file_path_resolved"])

    registry_base = {
        "model_id": model_id,
        "dataset_name": dataset_name,
        "dataset_short": plan_row["dataset_short"],
        "algorithm_name": ALGORITHM_NAME,
        "algorithm_short": ALGORITHM_SHORT,
        "patch_set": plan_row["patch_set"],
        "patch_size": plan_row["patch_size"],
        "ratio_variant": plan_row["ratio_variant"],
        "feature_set": plan_row["feature_set"],
        "feature_family": plan_row["feature_family"],
        "selection_group": plan_row["selection_group"],
        "selection_reason": plan_row["selection_reason"],
        "source_feature_file_path": plan_row["source_feature_file_path"],
        "source_feature_file_path_resolved": relative_path(feature_path),
        "source_linear_svm_model_path": plan_row["source_linear_svm_model_path"],
        "source_linear_svm_model_path_resolved": plan_row["source_linear_svm_model_path_resolved"],
        "model_file_path": relative_path(model_path),
        "model_fit_split": TRAIN_SPLIT,
        "valid_used_for_model_selection": True,
        "test_used_for_fit_or_selection": False,
        "dataset3_used": False,
        "detection_used": False,
        "class_weight": "None",
        "hyperparameter_grid_name": HYPERPARAMETER_GRID_NAME,
        "search_strategy": "train_split_gridsearch_cv_then_valid_selection",
        "cv_used": True,
        "cv_folds": CV_FOLDS,
        "cv_scoring": SCORING,
        "valid_selection_metric": "valid_f1",
        "score_method": SCORE_METHOD,
        "random_state": RANDOM_STATE,
        "created_at": timestamp_now(),
    }

    if model_path.exists() and not OVERWRITE_MODELS:
                return {
            **registry_base,
            "fit_status": "loaded_existing",
            "fit_runtime_seconds": time.time() - start_time,
            "selected_params_json": None,
            "best_cv_score": np.nan,
            "error_message": "",
            "notes": "Existing model artifact found and OVERWRITE_MODELS=False.",
        }

    if not feature_path.exists():
        return {
            **registry_base,
            "fit_status": "failed",
            "fit_runtime_seconds": time.time() - start_time,
            "error_message": f"Feature file not found: {feature_path}",
            "notes": "Missing feature file.",
        }

    try:
        feature_df = pd.read_parquet(feature_path)

        if SPLIT_COLUMN not in feature_df.columns:
            raise ValueError(f"Missing split column: {SPLIT_COLUMN}")
        if LABEL_COLUMN not in feature_df.columns:
            raise ValueError(f"Missing label column: {LABEL_COLUMN}")

        train_df = feature_df[feature_df[SPLIT_COLUMN] == TRAIN_SPLIT].copy()
        valid_df = feature_df[feature_df[SPLIT_COLUMN] == VALID_SPLIT].copy()

        if train_df.empty:
            raise ValueError("Train split is empty.")
        if valid_df.empty:
            raise ValueError("Valid split is empty.")

        feature_columns = get_feature_columns(feature_df)

        X_train = train_df[feature_columns].replace([np.inf, -np.inf], np.nan).fillna(0)
        y_train = train_df[LABEL_COLUMN].astype(int)

        X_valid = valid_df[feature_columns].replace([np.inf, -np.inf], np.nan).fillna(0)
        y_valid = valid_df[LABEL_COLUMN].astype(int)

        grid = GridSearchCV(
            estimator=make_estimator(),
            param_grid=PARAM_GRID,
            scoring=SCORING,
            cv=CV_FOLDS,
            n_jobs=N_JOBS,
            refit=True,
            return_train_score=True,
        )

        grid.fit(X_train, y_train)
        best_model = grid.best_estimator_

        y_valid_pred = best_model.predict(X_valid)
        valid_scores = extract_model_score(best_model, X_valid)

        metrics = compute_binary_metrics(y_valid, y_valid_pred, score_values=valid_scores)

        model_payload = {
            "model_id": model_id,
            "algorithm_name": ALGORITHM_NAME,
            "algorithm_short": ALGORITHM_SHORT,
            "dataset_name": dataset_name,
            "patch_set": plan_row["patch_set"],
            "feature_set": plan_row["feature_set"],
            "feature_columns": feature_columns,
            "estimator": best_model,
            "best_params": grid.best_params_,
            "best_cv_score": float(grid.best_score_),
            "valid_metrics": metrics,
            "source_feature_file_path": relative_path(feature_path),
            "source_linear_svm_model_path": plan_row["source_linear_svm_model_path_resolved"],
            "created_at": timestamp_now(),
        }

        save_model(model_payload, model_path, overwrite=OVERWRITE_MODELS, note=f"Stage 06 selected model: {model_id}")

        cv_results_df = pd.DataFrame(grid.cv_results_)
        cv_results_path = CV_RESULTS_DIR / f"{model_id}__cv_results.csv"
        save_dataframe_csv(cv_results_df, cv_results_path, overwrite=OVERWRITE_TABLES)

        return {
            **registry_base,
            "fit_status": "trained",
            "fit_runtime_seconds": time.time() - start_time,
            "selected_params_json": json.dumps(grid.best_params_, ensure_ascii=False),
            "best_cv_score": float(grid.best_score_),
            "feature_count_actual": len(feature_columns),
            "train_row_count_actual": len(train_df),
            "valid_row_count_actual": len(valid_df),
            "train_positive_count_actual": int((y_train == 1).sum()),
            "train_negative_count_actual": int((y_train == 0).sum()),
            "valid_positive_count_actual": int((y_valid == 1).sum()),
            "valid_negative_count_actual": int((y_valid == 0).sum()),
            **metrics,
            "error_message": "",
            "notes": "Model trained on train split and evaluated on valid split. Test split was not used.",
        }

    except Exception as exc:
        log_output(f"FAILED: {model_id}\n{traceback.format_exc()}")
        return {
            **registry_base,
            "fit_status": "failed",
            "fit_runtime_seconds": time.time() - start_time,
            "error_message": str(exc),
            "notes": traceback.format_exc(),
        }
    finally:
        gc.collect()


## 9. Seçili Modelleri Eğitme veya Yükleme

Her seçili kombinasyon için model dosyası kontrol edilir. Model mevcutsa ve overwrite kapalıysa dosya korunur; aksi durumda model eğitilir.


In [ ]:
registry_records = []

for row_index, plan_row in training_plan.iterrows():
    log_section(f"Training {row_index + 1}/{len(training_plan)} — {plan_row['model_id']}")
    result = train_one_selected_model(plan_row)
    registry_records.append(result)

registry_df = pd.DataFrame(registry_records)
log_dataframe("Stage 06 Training Registry", registry_df, max_rows=50)

save_dataframe_csv(registry_df, TABLES_DIR / "model_registry.csv", overwrite=OVERWRITE_TABLES)

status_summary = pd.DataFrame([
    {
        "algorithm_name": ALGORITHM_NAME,
        "expected_model_count": len(training_plan),
        "trained_count": int((registry_df["fit_status"] == "trained").sum()) if not registry_df.empty else 0,
        "loaded_existing_count": int((registry_df["fit_status"] == "loaded_existing").sum()) if not registry_df.empty else 0,
        "failed_count": int((registry_df["fit_status"] == "failed").sum()) if not registry_df.empty else 0,
        "debug_mode": DEBUG_MODE,
        "status": (
            "READY_FOR_REVIEW"
            if not registry_df.empty and int((registry_df["fit_status"] == "failed").sum()) == 0
            else "COMPLETED_WITH_FAILURES"
        ),
        "created_at": timestamp_now(),
    }
])

log_dataframe("Notebook Status", status_summary)
save_dataframe_csv(status_summary, TABLES_DIR / "notebook_status.csv", overwrite=OVERWRITE_TABLES)


## 10. Dataset Registry Tabloları

Aktif dataset için model registry ve sıralı validasyon tablosu model klasörü altına kaydedilir.


In [ ]:
for dataset_name in ACTIVE_DATASETS:
    dataset_registry = registry_df[registry_df["dataset_name"] == dataset_name].copy()
    model_root = MODEL_OUTPUT_ROOTS[dataset_name]

    save_dataframe_csv(
        dataset_registry,
        CV_RESULTS_DIR / "model_registry.csv",
        overwrite=OVERWRITE_TABLES,
        note=f"{dataset_name} {ALGORITHM_DISPLAY_NAME} model registry",
    )

    ranked_df = dataset_registry.copy()
    if "valid_f1" in ranked_df.columns:
        ranked_df = ranked_df.sort_values(
            ["valid_f1", "valid_recall", "valid_precision"],
            ascending=False,
            na_position="last",
        )

    save_dataframe_csv(
        ranked_df,
        CV_RESULTS_DIR / "ranked_valid_models.csv",
        overwrite=OVERWRITE_TABLES,
        note=f"{dataset_name} ranked {ALGORITHM_DISPLAY_NAME} validation models",
    )

log_output("Per-dataset registries saved.")


## 11. Final Durum

Notebook sonunda eğitim durumu kısa olarak özetlenir.


In [ ]:
log_section("12 FINAL DURUM")

status = "READY_FOR_REVIEW"
if "failed_count" in status_summary.columns and int(status_summary["failed_count"].iloc[0]) > 0:
    status = "COMPLETED_WITH_FAILURES"

final_status_df = pd.DataFrame([{
    "notebook_name": NOTEBOOK_NAME,
    "algorithm_name": ALGORITHM_NAME,
    "dataset_name": ACTIVE_DATASETS[0],
    "processed_model_count": len(training_plan),
    "status": status,
    "finished_at": timestamp_now(),
}])

final_status_df = save_dataframe_csv(
    final_status_df,
    TABLES_DIR / "final_status.csv",
    overwrite=OVERWRITE_TABLES,
    note="Final notebook status",
)
log_dataframe("Final Status", final_status_df)
log_output(f"Status: {status}")
log_output(f"Notebook tables directory: {TABLES_DIR}")
log_output(f"CV results directory: {CV_RESULTS_DIR}")
log_output("Selected model artifacts are stored in selected_classification_model_pool.")